# 第 7 章 销售预测与库存备货

本章商业问题：**下个月应该备多少货，如何减少缺货和积压？**

本章所有代码默认优先读取真实 ETL 接口 `http://192.168.31.47:38173/api/etl`。如果接口暂时不可用，会自动回退到本地 SQLite 后备数据，保证课堂可以继续进行。

## 0. 先建立商业问题意识

在商业课里，代码不是第一步。第一步是判断：这个问题影响收入、成本、用户体验、库存风险，还是营销效率？只有先说清楚商业目标，后面的数据选择和模型选择才不会跑偏。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if COURSE_ROOT.name in ["notebooks", "student_notebooks", "teacher_notebooks"]:
    COURSE_ROOT = COURSE_ROOT.parent
elif not (COURSE_ROOT / "course_utils").exists():
    COURSE_ROOT = Path("..").resolve()

sys.path.insert(0, str(COURSE_ROOT))

from course_utils.data_loader import (
    API_BASE, load_table, get_metrics, get_quality_report,
    get_table_catalog, get_schema, paid_orders, api_status, query_table
)
from course_utils.business import money, pct, section

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

print("课程目录:", COURSE_ROOT)
print("ETL API:", API_BASE)
print("API 状态:", api_status())

## 1. 查看 ETL 数据资产

下面先查看 ETL 接口暴露了哪些表。请注意 `dim_` 开头的是维度表，通常描述对象；`fact_` 开头的是事实表，通常记录业务事件。

In [ ]:
catalog = get_table_catalog()
tables = catalog["tables"]
print("可用表数量:", catalog.get("total", len(tables)))
for t in tables[:12]:
    print(t["tableName"], t.get("recordCount"), t.get("type"), t.get("description", ""))

In [ ]:
import pandas as pd
orders = paid_orders()
daily = orders.groupby("order_date").agg(sales=("paid_amount", "sum"), orders=("order_id", "nunique")).reset_index().sort_values("order_date")
daily = daily.set_index("order_date").asfreq("D").fillna(0)
daily["ma7"] = daily["sales"].rolling(7).mean()
daily["ma30"] = daily["sales"].rolling(30).mean()
daily.tail()

In [ ]:
import matplotlib.pyplot as plt
daily[["sales", "ma7", "ma30"]].tail(120).plot(figsize=(11, 4), title="Recent Sales and Moving Average")
plt.tight_layout()

In [ ]:
recent = daily["sales"].tail(30)
base_forecast = recent.mean()
safety = recent.std() * 1.65
print("基准日销售预测:", money(base_forecast))
print("安全库存金额建议:", money(safety))

## 商业解释

销售预测的重点不是预测到小数点，而是帮助供应链决定备货区间。移动平均适合入门教学，因为它让学生直观看到近期需求基线。

In [ ]:
assert len(daily) > 30
print("第 07 章验证通过")